In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
print('LLM ready')

/Users/rahultiwari/Documents/02_Freelancing/coding_ninja_fresh/dummy-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM ready


In [2]:
GOOD_PROMPT = ChatPromptTemplate.from_template("""
You are BajajBot, an AI assistant for Bajaj Finance helpdesk agents.

CONTEXT (retrieved from Bajaj Finance policy documents):
-------------------------------------------------------
{context}
-------------------------------------------------------

INSTRUCTIONS:
- Answer ONLY using the CONTEXT above.
- Do NOT use your training knowledge.
- If the answer is not in the CONTEXT, say exactly:
  "I don't have this information in the provided documents."
- Keep your answer under 3 sentences.
- If a specific number or rule is in CONTEXT, include it.

QUESTION: {question}

ANSWER:
""")

In [3]:
chain = GOOD_PROMPT | llm | StrOutputParser()


In [8]:
# pip install langchain-community

In [9]:
# pip install pypdf

In [4]:
PDF_PATH = 'data/bajaj_finance_policy_reference.pdf'

from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader(PDF_PATH)
pages  = loader.load()

In [12]:
len(pages)

12

In [5]:
# pages[0].page_content.split('\n')

In [ ]:
# next big question
len(pages[0].page_content)

# splitting == chunks ??


1153

In [21]:
SAMPLE_TEXT="""GOLD LOAN POLICY — BAJAJ FINANCE

Gold purity accepted: 18 karat to 24 karat.
Maximum LTV: 75 percent of market value (RBI mandated).
Tenure: 3 months to 24 months.

Foreclosure Policy:
Within 3 months: 2 percent foreclosure charge
3 to 6 months: 2 percent charge
6 to 12 months: 1 percent charge
After 12 months: No foreclosure charge

Disbursal: Same day within 30 minutes of appraisal.
"""


In [23]:
chunk_sixe = 200
chunks = [SAMPLE_TEXT[i:i+chunk_sixe] for i in range(0, len(SAMPLE_TEXT), chunk_sixe)]
chunks[0]

'GOLD LOAN POLICY — BAJAJ FINANCE\n\nGold purity accepted: 18 karat to 24 karat.\nMaximum LTV: 75 percent of market value (RBI mandated).\nTenure: 3 months to 24 months.\n\nForeclosure Policy:\nWithin 3 month'

In [24]:
chunks[1]

's: 2 percent foreclosure charge\n3 to 6 months: 2 percent charge\n6 to 12 months: 1 percent charge\nAfter 12 months: No foreclosure charge\n\nDisbursal: Same day within 30 minutes of appraisal.\n'

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a text splitter
# chunk_size = maximum size of one chunk
# chunk_overlap = small repeated part between two nearby chunks
# separators = order in which LangChain tries to split the text
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len, #length_function=len tells the splitter how to measure the size of text
)

chunks = splitter.split_documents(pages)

In [7]:
len(chunks[0].page_content)

512

In [8]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9288.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
chunks[0].page_content

'BAJAJ FINANCE LIMITED\nInternal Policy Reference Document — CONFIDENTIAL\nPage 1\nFor internal use only | Bajaj Finance Limited | FY 2024-25 | Unauthorised distribution prohibited\n BAJAJ FINANCE LIMITED\n Policy & Procedure Reference\n Document\n Helpdesk Agent Knowledge Base\n FY 2024–25 | All Loan Products\nDocument Type\nInternal Policy & Procedure Manual\nCoverage\nPersonal Loan · Home Loan · Gold Loan · Business Loan · CIBIL Policy\nIntended Users\nHelpdesk Agents · Branch Executives · Collections Team · Sales Team'

In [16]:
chunk_texts = [c.page_content for c in chunks]
chunk_meta  = [c.metadata     for c in chunks]

chunk_vectors = model.encode(
    chunk_texts,
    show_progress_bar = True,
    convert_to_numpy  = True,
    normalize_embeddings = True,   # makes cosine == dot product (faster)
)



Batches: 100%|██████████| 2/2 [00:00<00:00,  3.93it/s]


In [15]:
# chunk_vectors

len(chunk_vectors[0])

384

In [17]:
store = {
    'vectors' : chunk_vectors,
    'texts'   : chunk_texts,
    'meta'    : chunk_meta,
    'model'   : 'all-MiniLM-L6-v2',   # CRITICAL: remember which model produced these
}

import pickle

with open('bajaj_vectors_2.pkl', 'wb') as f:
    pickle.dump(store, f)

In [20]:
with open('bajaj_vectors_2.pkl', 'rb') as f:
    loaded = pickle.load(f)

chunk_vectors = loaded['vectors']
chunk_texts = loaded['texts']
chunk_meta = loaded['meta']
# chunk_vectors

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [26]:
from sklearn.metrics.pairwise import cosine_similarity


def search_similarity(query, top_k=3):
    # Convert the query into an embedding
    query_vector = model.encode([query], convert_to_numpy=True)

    # Compare query with all chunk vectors using cosine similarity
    scores = cosine_similarity(query_vector, chunk_vectors)[0]

    # Get indexes of top matching chunks
    top_indexes = scores.argsort()[::-1][:top_k]

    # Return (score, index)
    results = []
    for i in top_indexes:
        results.append((float(scores[i]), int(i)))

    return results



q_relevant = "What is the foreclosure charge on gold loans?"
q_irrelevant = "How do I bake a chocolate cake?"
search_similarity(q_irrelevant)

[(0.10273120552301407, 4),
 (0.09742630273103714, 26),
 (0.050641074776649475, 2)]

In [ ]:
# print(chunk_texts[4])

Age
21 – 60 years for salaried | 25 – 65 years for self-employed
Minimum Income
■25,000/month net take-home (salaried) | ■4.8 lakhs annual ITR (self-employed)
CIBIL Score
700 and above for standard approval | 650–699 subject to enhanced scrutiny
Employment
Minimum 1 year current employer | 2 years total work experience
FOIR Limit
Maximum 50% Fixed Obligation to Income Ratio including proposed EMI
Self-Employed
Minimum 3 years in same profession | Doctors, CAs, Architects eligible
Govt/Defence


In [28]:
def show(title, results, query):
    print("\n===", title, "===")
    print("Query:", query)
    print("Found:", len(results))

    for i, (score, idx) in enumerate(results, 1):
        text = chunk_texts[idx][:110].replace("\n", " ")
        page = chunk_meta[idx].get("page", "?")
        print(f"{i}. score={score:.4f} | page {page} | {text}...")

In [29]:
show("Similarity (relevant)", search_similarity(q_relevant), q_relevant)
show("Similarity (IRRELEVANT)", search_similarity(q_irrelevant), q_irrelevant)



=== Similarity (relevant) ===
Query: What is the foreclosure charge on gold loans?
Found: 3
1. score=0.7085 | page 8 | 7.2 Foreclosure Policy Gold loan foreclosure (early closure before tenure end) is governed by the following po...
2. score=0.6014 | page 8 | Maximum 75% of gold market value (RBI mandated ceiling) Tenure 3 months to 24 months | Bullet repayment or mon...
3. score=0.4401 | page 1 | ■1,000 + GST per bounced NACH/ECS mandate Prepayment (Full) 4% of outstanding principal if closed within 12 mo...

=== Similarity (IRRELEVANT) ===
Query: How do I bake a chocolate cake?
Found: 3
1. score=0.1027 | page 1 | Age 21 – 60 years for salaried | 25 – 65 years for self-employed Minimum Income ■25,000/month net take-home (s...
2. score=0.0974 | page 7 | 14% – 24% p.a. based on CIBIL and business vintage Collateral Not required for amounts up to ■50 lakhs | Requi...
3. score=0.0506 | page 0 | Page 7 07  Gold Loan — Products & Process  Page 8 08  CIBIL Score — Impact & Reporting  Page 9

In [30]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

GOOD_PROMPT = ChatPromptTemplate.from_template("""
You are BajajBot, an AI assistant for Bajaj Finance helpdesk agents.

CONTEXT (retrieved from Bajaj Finance policy documents):
-------------------------------------------------------
{context}
-------------------------------------------------------

INSTRUCTIONS:
- Answer ONLY using the CONTEXT above.
- Do NOT use your training knowledge.
- If the answer is not in the CONTEXT, say exactly:
  "I don't have this information in the provided documents."
- Keep your answer under 3 sentences.
- If a specific number or rule is in CONTEXT, include it.

QUESTION: {question}

ANSWER:
""")

In [31]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

chain = GOOD_PROMPT | llm | StrOutputParser()

In [32]:

def search_similarity(query, top_k=3):
    query_vector = model.encode([query], convert_to_numpy=True)
    scores = cosine_similarity(query_vector, chunk_vectors)[0]
    top_indexes = scores.argsort()[::-1][:top_k]

    results = []
    for i in top_indexes:
        results.append((float(scores[i]), int(i)))

    return results


def get_context(query, top_k=3):
    results = search_similarity(query, top_k=top_k)
    print(results)

    context_parts = []
    for score, idx in results:
        text = chunk_texts[idx]
        context_parts.append(text)

    print(context_parts)

    context = "\n\n".join(context_parts)
    return context

In [35]:
query = "How do I bake a chocolate cake?"

context = get_context(query, top_k=3)
context 

[(0.10273120552301407, 4), (0.09742630273103714, 26), (0.050641074776649475, 2)]
['Age\n21 – 60 years for salaried | 25 – 65 years for self-employed\nMinimum Income\n■25,000/month net take-home (salaried) | ■4.8 lakhs annual ITR (self-employed)\nCIBIL Score\n700 and above for standard approval | 650–699 subject to enhanced scrutiny\nEmployment\nMinimum 1 year current employer | 2 years total work experience\nFOIR Limit\nMaximum 50% Fixed Obligation to Income Ratio including proposed EMI\nSelf-Employed\nMinimum 3 years in same profession | Doctors, CAs, Architects eligible\nGovt/Defence', '14% – 24% p.a. based on CIBIL and business vintage\nCollateral\nNot required for amounts up to ■50 lakhs | Required above ■50 lakhs\nBusiness Types\nSole Proprietorship | Partnership | Private Limited | LLP | Public Limited\nAnnual Turnover\nMinimum ■40 lakhs (unsecured) | Minimum ■1 crore (secured)\nBusiness Vintage\nMinimum 3 years in profit | 5 years for manufacturing sector', 'Page 7\n07\n Gold Lo

'Age\n21 – 60 years for salaried | 25 – 65 years for self-employed\nMinimum Income\n■25,000/month net take-home (salaried) | ■4.8 lakhs annual ITR (self-employed)\nCIBIL Score\n700 and above for standard approval | 650–699 subject to enhanced scrutiny\nEmployment\nMinimum 1 year current employer | 2 years total work experience\nFOIR Limit\nMaximum 50% Fixed Obligation to Income Ratio including proposed EMI\nSelf-Employed\nMinimum 3 years in same profession | Doctors, CAs, Architects eligible\nGovt/Defence\n\n14% – 24% p.a. based on CIBIL and business vintage\nCollateral\nNot required for amounts up to ■50 lakhs | Required above ■50 lakhs\nBusiness Types\nSole Proprietorship | Partnership | Private Limited | LLP | Public Limited\nAnnual Turnover\nMinimum ■40 lakhs (unsecured) | Minimum ■1 crore (secured)\nBusiness Vintage\nMinimum 3 years in profit | 5 years for manufacturing sector\n\nPage 7\n07\n Gold Loan — Products & Process\n Page 8\n08\n CIBIL Score — Impact & Reporting\n Page 9\n

In [36]:
query = "How do I bake a chocolate cake?"

context = get_context(query, top_k=3)

print("Context:\n")


answer = chain.invoke({
    "context": context,
    "question": query
})

print("\nAnswer:\n")
print(answer)

[(0.10273120552301407, 4), (0.09742630273103714, 26), (0.050641074776649475, 2)]
['Age\n21 – 60 years for salaried | 25 – 65 years for self-employed\nMinimum Income\n■25,000/month net take-home (salaried) | ■4.8 lakhs annual ITR (self-employed)\nCIBIL Score\n700 and above for standard approval | 650–699 subject to enhanced scrutiny\nEmployment\nMinimum 1 year current employer | 2 years total work experience\nFOIR Limit\nMaximum 50% Fixed Obligation to Income Ratio including proposed EMI\nSelf-Employed\nMinimum 3 years in same profession | Doctors, CAs, Architects eligible\nGovt/Defence', '14% – 24% p.a. based on CIBIL and business vintage\nCollateral\nNot required for amounts up to ■50 lakhs | Required above ■50 lakhs\nBusiness Types\nSole Proprietorship | Partnership | Private Limited | LLP | Public Limited\nAnnual Turnover\nMinimum ■40 lakhs (unsecured) | Minimum ■1 crore (secured)\nBusiness Vintage\nMinimum 3 years in profit | 5 years for manufacturing sector', 'Page 7\n07\n Gold Lo

In [ ]:
# The foreclosure charge on gold loans is as follows: 
# Within 3 months of disbursal, it is 2% on the outstanding principal 
# plus accrued interest; between 3 to 6 months, 
# it is 2% on the outstanding principal; between 6 to 12 months, 
# it is 1% on the outstanding principal; and after 12 months, 
# there is no foreclosure charge, and 
# the customer pays only the outstanding principal plus interest.
